In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as f

In [2]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("OlympicsDevelopment")
    .getOrCreate()
)

spark

In [3]:
DATA_ROOT = Path("../data/test_output")

#### Imports and loaded tables

In [4]:

bronze_athletes = spark.read.parquet(f'{DATA_ROOT}/bronze/athlete_events')
bronze_noc = spark.read.parquet(f'{DATA_ROOT}/bronze/noc_regions')
dim_athlete = spark.read.parquet(f'{DATA_ROOT}/gold/dim_athlete')
dim_games = spark.read.parquet(f'{DATA_ROOT}/gold/dim_games')
dim_event = spark.read.parquet(f'{DATA_ROOT}/gold/dim_event')
dim_noc = spark.read.parquet(f'{DATA_ROOT}/gold/dim_noc')
fact_participation = spark.read.parquet(f'{DATA_ROOT}/gold/fact_participation')


##### Confirm one current version per NOC

In [5]:
invalid_current_versions = (
    dim_noc
    .filter(f.col("is_current"))
    .groupBy("noc_code")
    .count()
    .filter(f.col("count") != 1)
)

invalid_current_versions.show()

+--------+-----+
|noc_code|count|
+--------+-----+
+--------+-----+



##### Confirm that fact foreign keys are populated

In [6]:
fac_key_quality = (
    fact_participation
    .select(
        f.sum(f.col('athlete_key').isNull().cast('int')).alias('null_athlete_keys'),
        f.sum(f.col('games_key').isNull().cast('int')).alias('null_games_keys'),
        f.sum(f.col('event_key').isNull().cast('int')).alias('null_event_keys'),
        f.sum(f.col('noc_key').isNull().cast('int')).alias('null_noc_keys')
    )
)

fac_key_quality.show()

+-----------------+---------------+---------------+-------------+
|null_athlete_keys|null_games_keys|null_event_keys|null_noc_keys|
+-----------------+---------------+---------------+-------------+
|                0|              0|              0|            0|
+-----------------+---------------+---------------+-------------+



##### Check if each FK have a match in the dim tables

In [ ]:
missing_athlete_keys = (
    fact_participation
    .select("athlete_key").distinct()
    .join(
        dim_athlete.select("athlete_key").distinct(),
        on="athlete_key",
        how="left_anti"
    )
)

missing_games_keys = (
    fact_participation
    .select("games_key").distinct()
    .join(
        dim_games.select("games_key").distinct(),
        on="games_key",
        how="left_anti"
    )
)

missing_event_keys = (
    fact_participation
    .select("event_key").distinct()
    .join(
        dim_event.select("event_key").distinct(),
        on="event_key",
        how="left_anti"
    )
)

missing_noc_keys = (
    fact_participation
    .select("noc_key").distinct()
    .join(
        dim_noc.select("noc_key").distinct(),
        on="noc_key",
        how="left_anti"
    )
)

print(f"Missing athlete keys: {missing_athlete_keys.count()}")
print(f"Missing games keys: {missing_games_keys.count()}")
print(f"Missing event keys: {missing_event_keys.count()}")
print(f"Missing NOC keys: {missing_noc_keys.count()}")


Missing athlete keys: 0
Missing games keys: 0
Missing event keys: 0
Missing NOC keys: 0


In [10]:
spark.stop()